In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

CHEMIN_PROCESSED = "../data/processed/"
CHEMIN_MODELES = "../models/"

In [2]:
X_test = pd.read_csv(f"{CHEMIN_PROCESSED}X_test.csv")
y_test = pd.read_csv(f"{CHEMIN_PROCESSED}y_test.csv").squeeze()

print("X_test shape:", X_test.shape)
print(X_test.head())

X_test shape: (3737, 46)
    hp  retreat_cost  has_evolution  generation  pokedex_number  \
0   80             2              1         2.0           205.0   
1  150             4              1         1.0            34.0   
2   60             1              0         5.0           517.0   
3   50             1              0         1.0             4.0   
4    0             0              0        -1.0            -1.0   

   carte_age_ans  set_total_cartes  position_dans_set  set_serie_encoded  \
0             16                91              0.143                  5   
1             10               116              0.388                 14   
2             15                12              0.583                  8   
3             24               110              0.636                  8   
4             23               182              0.681                  3   

   supertype_encoded  ...  a_attaque_100plus  a_attaque_200plus  \
0                  1  ...                  0    

In [3]:
resultats = []

for nom in ["ridge", "random_forest", "xgboost"]:
    modele = joblib.load(f"{CHEMIN_MODELES}{nom}.pkl")
    y_pred = modele.predict(X_test)

    y_reel = np.expm1(y_test)
    y_pred_reel = np.expm1(y_pred)

    resultats.append({
        "modele":   nom,
        "R²":       round(r2_score(y_test, y_pred), 4),
        "RMSE ($)": round(np.sqrt(mean_squared_error(y_reel, y_pred_reel)), 2),
        "MAE ($)":  round(mean_absolute_error(y_reel, y_pred_reel), 2),
    })

df_resultats = pd.DataFrame(resultats).sort_values("R²", ascending=False)
print(df_resultats.to_string(index=False))
print(f"\nmeilleur modele : {df_resultats.iloc[0]['modele']}")

       modele     R²  RMSE ($)  MAE ($)
      xgboost 0.8529     49.35     8.69
random_forest 0.8349     50.59     9.22
        ridge 0.5472     64.18    12.98

meilleur modele : xgboost
